# Random Forest — Análise Completa
**Universidade Estadual Paulista "Júlio de Mesquita Filho" — UNESP ICT**  
**Disciplina:** Reconhecimento de Padrões  
**Referência:** Prof. Dr. Rogério Galante Negri

---

Este notebook apresenta uma análise completa do método **Random Forest**, cobrindo:
- Fundamentos dos métodos de *ensemble*
- CART para classificação e regressão
- Técnicas aplicáveis ao Random Forest
- Pré-processamento, avaliação e parametrização
- Implementação prática com scikit-learn

## 0. Dependências

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification, make_regression, load_iris
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    BaggingClassifier, AdaBoostClassifier, StackingClassifier
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    cohen_kappa_score, f1_score, matthews_corrcoef,
    ConfusionMatrixDisplay
)
from scipy.stats import loguniform, uniform

np.random.seed(42)
print('Dependências carregadas com sucesso.')

---
## 1. Métodos de Ensemble — Contexto

Métodos de *ensemble* combinam múltiplos classificadores para obter resultados mais robustos e com menor taxa de erro do que classificadores isolados. O Random Forest é um caso especial de ensemble.

As principais estratégias de combinação estudadas são:

| Método | Diversidade via | Combinação | Sequencial? |
|---|---|---|---|
| Majority Voting | Classificadores distintos | Votação | Não |
| Stacking | Modelos distintos | Meta-classificador | Não |
| Bagging | Bootstrap | Majority Voting | Não |
| AdaBoost | Pesos adaptativos | Votos ponderados | Sim |
| **Random Forest** | Bootstrap + atributos aleatórios | Majority Voting | Não |

### 1.1 Regras de Combinação baseadas em Bayes

A partir da regra de Bayes, derivam-se cinco regras de combinação para $L$ classificadores:

| Regra | Expressão |
|---|---|
| **Produto** | $\arg\max_j \prod_{i=1}^{L} p(x^{(i)}|\omega_j)$ |
| **Soma** | $\arg\max_j \sum_{i=1}^{L} p(x^{(i)}|\omega_j) \cdot w_i$ |
| **Máximo** | $\max_j\{p(\omega_j|x^{(i)})\} > \max_j\{p(\omega_k|x^{(i)})\}$ |
| **Mínimo** | $\min_j\{p(\omega_j|x^{(i)})\} > \min_j\{p(\omega_k|x^{(i)})\}$ |
| **Mediana** | $\text{med}_j\{p(\omega_j|x^{(i)})\} > \text{med}_j\{p(\omega_k|x^{(i)})\}$ |

A **Regra da Soma** é a mais robusta por suportar pesos $w_i$ e ser mais estável que a Regra do Produto em presença de erros de estimação.

In [ ]:
# Demonstração das regras de combinação
# Probabilidades a posteriori de 3 classificadores para 3 classes
posteriors = np.array([
    [0.6, 0.3, 0.1],   # classificador 1
    [0.4, 0.5, 0.1],   # classificador 2
    [0.5, 0.3, 0.2],   # classificador 3
])

classes = ['ω1', 'ω2', 'ω3']

produto = np.prod(posteriors, axis=0)
soma    = np.sum(posteriors, axis=0)
maximo  = np.max(posteriors, axis=0)
minimo  = np.min(posteriors, axis=0)
mediana = np.median(posteriors, axis=0)

print(f"{'Regra':<12} {'ω1':>8} {'ω2':>8} {'ω3':>8} {'Decisão':>10}")
print('-' * 52)
for nome, scores in [('Produto', produto), ('Soma', soma),
                      ('Máximo', maximo), ('Mínimo', minimo), ('Mediana', mediana)]:
    print(f"{nome:<12} {scores[0]:>8.4f} {scores[1]:>8.4f} {scores[2]:>8.4f} {classes[np.argmax(scores)]:>10}")

### 1.2 Majority Voting

In [ ]:
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                            n_clusters_per_class=1, n_classes=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

from sklearn.ensemble import VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

clf1 = DecisionTreeClassifier(max_depth=3, random_state=42)
clf2 = GaussianNB()
clf3 = KNeighborsClassifier(n_neighbors=5)

voting = VotingClassifier(estimators=[('dt', clf1), ('nb', clf2), ('knn', clf3)],
                           voting='soft')

for nome, clf in [('Decision Tree', clf1), ('Naive Bayes', clf2),
                   ('KNN', clf3), ('Majority Voting', voting)]:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"{nome:<20}: Acurácia = {acc:.4f}")

### 1.3 Bagging vs AdaBoost

In [ ]:
bagging  = BaggingClassifier(estimator=DecisionTreeClassifier(),
                              n_estimators=50, random_state=42)
adaboost = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                               n_estimators=50, algorithm='SAMME', random_state=42)

for nome, clf in [('Bagging', bagging), ('AdaBoost/SAMME', adaboost)]:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"{nome:<20}: Acurácia = {acc:.4f}")

---
## 2. CART — Classification and Regression Trees

O CART é o **base learner** do Random Forest. Funciona recursivamente dividindo o espaço de atributos por limiares $\tau_{kh}$.

### 2.1 CART para Classificação

**Impureza** medida pela Entropia de Informação:

$$I(\mathcal{Q}) = -\sum_{j=1}^{c} P(\omega_j|\mathcal{Q}) \cdot \log_2 P(\omega_j|\mathcal{Q})$$

**Redução de impureza** pelo limiar $\tau_{kh}$:

$$\Delta I(\mathcal{Q};\tau_{kh}) = I(\mathcal{Q}) - \frac{\#\mathcal{Q}_{inf}}{\#\mathcal{Q}}I(\mathcal{Q}_{inf}(\tau_{kh})) - \frac{\#\mathcal{Q}_{sup}}{\#\mathcal{Q}}I(\mathcal{Q}_{sup}(\tau_{kh}))$$

Divisão aceita quando $\Delta I > \zeta$ e $\#\mathcal{Q} > \psi$. Folha rotulada com:

$$\omega^* = \arg\max_{\omega_j \in \Omega} P(\omega_j|\mathcal{Q})$$

In [ ]:
# Implementação manual da Entropia de Informação
def entropia(y):
    classes, contagens = np.unique(y, return_counts=True)
    p = contagens / len(y)
    return -np.sum(p * np.log2(p + 1e-12))

def reducao_impureza(y, y_esq, y_dir):
    n = len(y)
    return entropia(y) - (len(y_esq)/n)*entropia(y_esq) - (len(y_dir)/n)*entropia(y_dir)

# Demonstração com iris
iris = load_iris()
X_iris, y_iris = iris.data[:, :2], iris.target

print(f"Entropia total do dataset Iris (2 atributos): {entropia(y_iris):.4f} bits")
print(f"Entropia máxima (classes equiprováveis, c=3): {np.log2(3):.4f} bits")

In [ ]:
# Efeito do parâmetro ψ (min_samples_leaf) na superfície de decisão
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

h = 0.02
x_min, x_max = X_iris[:, 0].min() - 0.5, X_iris[:, 0].max() + 0.5
y_min, y_max = X_iris[:, 1].min() - 0.5, X_iris[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold  = ['red', 'green', 'blue']

for ax, psi in zip(axes, [1, 5, 20]):
    cart = DecisionTreeClassifier(min_samples_leaf=psi, criterion='entropy', random_state=42)
    cart.fit(X_iris, y_iris)
    Z = cart.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    for cls, cor in enumerate(cmap_bold):
        ax.scatter(X_iris[y_iris==cls, 0], X_iris[y_iris==cls, 1],
                   c=cor, s=20, label=iris.target_names[cls])
    acc = accuracy_score(y_iris, cart.predict(X_iris))
    ax.set_title(f'CART — ψ = {psi}\nAcurácia = {acc:.2%}')
    ax.set_xlabel('Comprimento Sépala')
    ax.set_ylabel('Largura Sépala')

axes[0].legend()
plt.suptitle('Efeito do parâmetro ψ (min_samples_leaf) no CART', fontsize=13)
plt.tight_layout()
plt.show()

### 2.2 CART para Regressão

A impureza é medida pelo **Desvio Quadrático**:

$$D(\mathcal{Q}) = \sum_{i:(\mathbf{x}_i, y_i) \in \mathcal{Q}} (y_i - \bar{y}_{\mathcal{Q}})^2 \qquad \bar{y}_{\mathcal{Q}} = \frac{1}{\#\mathcal{Q}}\sum y_i$$

A predição em cada folha é a **média** $\bar{y}_{\mathcal{Q}}$, resultando em uma função por partes (*piecewise constant*).

In [ ]:
np.random.seed(42)
X_reg = np.sort(np.random.uniform(0, 6, 100)).reshape(-1, 1)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.2, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
X_plot = np.linspace(0, 6, 300).reshape(-1, 1)

for ax, psi in zip(axes, [1, 5, 20]):
    cart_r = DecisionTreeRegressor(min_samples_leaf=psi, random_state=42)
    cart_r.fit(X_reg, y_reg)
    ax.scatter(X_reg, y_reg, s=10, alpha=0.6, label='Dados')
    ax.plot(X_plot, cart_r.predict(X_plot), 'r-', lw=2, label='CART')
    ax.plot(X_plot, np.sin(X_plot), 'g--', lw=1.5, label='sin(x)')
    ax.set_title(f'CART Regressão — ψ = {psi}')
    ax.set_xlabel('x')
    ax.legend(fontsize=8)

plt.suptitle('CART Regressão: função por partes vs. ψ', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Random Forest

### 3.1 Definição e Pilares

O Random Forest combina $L$ árvores CART independentes, cada uma treinada em uma réplica bootstrap do dataset. O diferencial em relação ao Bagging é a **seleção aleatória de atributos** em cada nó:

```
D → {D₁, D₂, ..., DL}        (bootstrap)
X^(l) ⊆ X por nó              (aleatorização de atributos)
g₁, g₂, ..., gL               (árvores independentes)
g* = Majority Voting(g₁...gL) (decisão final)
```

| Aspecto | Bagging | Random Forest |
|---|---|---|
| Amostragem dos dados | Bootstrap | Bootstrap |
| Seleção de atributos | Todos | Subconjunto aleatório ($\sqrt{p}$ ou $\log_2 p$) |
| Combinação | Majority Voting | Majority Voting |
| Diversidade | Moderada | **Alta** |
| Base learner | Qualquer | Árvores CART |

In [ ]:
# Comparação: Decision Tree × Bagging × Random Forest
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                            n_redundant=2, n_classes=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

modelos = {
    'CART (árvore simples)': DecisionTreeClassifier(criterion='entropy', random_state=42),
    'Bagging (50 árvores)' : BaggingClassifier(estimator=DecisionTreeClassifier(),
                                                n_estimators=50, random_state=42),
    'Random Forest (50)'   : RandomForestClassifier(n_estimators=50, random_state=42),
}

print(f"{'Modelo':<30} {'Treino':>8} {'Teste':>8} {'Kappa':>8}")
print('-' * 56)
for nome, clf in modelos.items():
    clf.fit(X_train, y_train)
    acc_tr = accuracy_score(y_train, clf.predict(X_train))
    acc_te = accuracy_score(y_test,  clf.predict(X_test))
    kappa  = cohen_kappa_score(y_test, clf.predict(X_test))
    print(f"{nome:<30} {acc_tr:>8.4f} {acc_te:>8.4f} {kappa:>8.4f}")

In [ ]:
# Efeito do número de árvores L
L_values = [1, 5, 10, 25, 50, 100, 200]
accs_rf  = []
accs_bag = []

for L in L_values:
    rf  = RandomForestClassifier(n_estimators=L, random_state=42).fit(X_train, y_train)
    bag = BaggingClassifier(estimator=DecisionTreeClassifier(),
                             n_estimators=L, random_state=42).fit(X_train, y_train)
    accs_rf.append(accuracy_score(y_test, rf.predict(X_test)))
    accs_bag.append(accuracy_score(y_test, bag.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(L_values, accs_rf,  'b-o', label='Random Forest')
plt.plot(L_values, accs_bag, 'r--s', label='Bagging')
plt.xlabel('Número de árvores (L)')
plt.ylabel('Acurácia no teste')
plt.title('Acurácia vs. Número de Árvores')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Técnicas Aplicáveis ao Random Forest

### 4.1 Out-of-Bag (OOB) Estimation

Por ser treinado via bootstrap, cada árvore **não vê** ~37% dos exemplos. Esses exemplos *out-of-bag* permitem estimar o erro de generalização sem conjunto de validação separado.

In [ ]:
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_oob.fit(X_train, y_train)

acc_teste = accuracy_score(y_test, rf_oob.predict(X_test))
print(f'Acurácia OOB    : {rf_oob.oob_score_:.4f}')
print(f'Acurácia no Teste: {acc_teste:.4f}')
print(f'Diferença        : {abs(rf_oob.oob_score_ - acc_teste):.4f}')

### 4.2 Importância de Atributos

O Random Forest calcula a **importância de cada atributo** como a redução média de impureza (Gini/Entropia) proporcionada por aquele atributo ao longo de todas as árvores.

In [ ]:
importancias = rf_oob.feature_importances_
indices = np.argsort(importancias)[::-1]
n_feat  = X_train.shape[1]

plt.figure(figsize=(8, 4))
plt.bar(range(n_feat), importancias[indices], color='steelblue', alpha=0.8)
plt.xticks(range(n_feat), [f'X{i+1}' for i in indices])
plt.xlabel('Atributo')
plt.ylabel('Importância (redução média de impureza)')
plt.title('Importância de Atributos — Random Forest')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Ranking de importância:')
for rank, i in enumerate(indices):
    print(f'  {rank+1}. X{i+1}: {importancias[i]:.4f}')

### 4.3 Regra da Soma Ponderada

Em vez do Majority Voting simples, as árvores podem contribuir com **pesos** $w_i$ proporcionais ao seu desempenho no OOB.

In [ ]:
# Simulação: votação ponderada pelo desempenho OOB individual de cada árvore
rf_100 = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_100.fit(X_train, y_train)

# Predições por soft-voting (probabilidades médias = regra da soma uniforme)
proba_media = rf_100.predict_proba(X_test)
pred_soma   = np.argmax(proba_media, axis=1)

acc_mv   = accuracy_score(y_test, rf_100.predict(X_test))
acc_soma = accuracy_score(y_test, pred_soma)

print(f'Majority Voting (hard) : {acc_mv:.4f}')
print(f'Regra da Soma (soft)   : {acc_soma:.4f}')
print('(soft-voting usa probabilidades médias — mais informativo que votos binários)')

### 4.4 Stacking com Random Forest como Base Learner

In [ ]:
base_learners = [
    ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
    ('svm', SVC(probability=True, kernel='rbf', C=10, random_state=42)),
]

stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

stacking.fit(X_train, y_train)
acc_stack = accuracy_score(y_test, stacking.predict(X_test))
print(f'Stacking (RF + SVM → LR): Acurácia = {acc_stack:.4f}')

---
## 5. Pré-processamento para o Random Forest

### 5.1 Maldição da Dimensionalidade

Em espaços de alta dimensão, qualquer padrão tende a ser distante de todos os outros. O RF mitiga isso internamente via aleatorização de atributos, mas técnicas externas de redução dimensional podem ser aplicadas como pré-processamento.

In [ ]:
# Demonstração: distâncias médias em dimensões crescentes
dims   = [2, 10, 50, 100, 200, 500, 1000]
dists  = []
for d in dims:
    pts = np.random.uniform(0, 1, (1000, d))
    # distância entre pontos consecutivos
    d_med = np.mean(np.linalg.norm(pts[:-1] - pts[1:], axis=1))
    dists.append(d_med)

plt.figure(figsize=(7, 4))
plt.plot(dims, dists, 'b-o')
plt.xlabel('Dimensão do espaço')
plt.ylabel('Distância média entre pontos')
plt.title('Maldição da Dimensionalidade')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 PCA — Principal Component Analysis

Realiza rotação dos eixos do espaço original, gerando componentes ortogonais ordenados pela variância:

$$\mathbf{x}' = \mathbf{W}^T(\mathbf{x} - \boldsymbol{\mu}) \qquad \mathbf{x}'_p = \mathbf{W}_p^T(\mathbf{x} - \boldsymbol{\mu})$$

In [ ]:
# PCA + Random Forest
X_hd, y_hd = make_classification(n_samples=400, n_features=30, n_informative=8,
                                   n_redundant=5, n_classes=3, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_hd, y_hd, test_size=0.3, random_state=42)

pca = PCA(n_components=0.95)   # retém 95% da variância
X_tr_pca = pca.fit_transform(X_tr)
X_te_pca = pca.transform(X_te)

rf_orig = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
rf_pca  = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr_pca, y_te[:len(X_tr_pca)])
rf_pca2 = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr_pca, y_tr)

print(f'Dimensão original       : {X_tr.shape[1]} atributos')
print(f'Dimensão após PCA (95%) : {X_tr_pca.shape[1]} componentes')
print(f'RF sem PCA — Acurácia   : {accuracy_score(y_te, rf_orig.predict(X_te)):.4f}')
print(f'RF com PCA — Acurácia   : {accuracy_score(y_te, rf_pca2.predict(X_te_pca)):.4f}')

# Variância explicada acumulada
pca_full = PCA().fit(X_tr)
plt.figure(figsize=(7, 4))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), 'b-o', ms=4)
plt.axhline(0.95, color='r', linestyle='--', label='95% variância')
plt.xlabel('Número de componentes')
plt.ylabel('Variância explicada acumulada')
plt.title('PCA — Variância Explicada')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Seleção de Atributos — SFS e SBS

| Algoritmo | Estratégia | Critério |
|---|---|---|
| **SFS** | *Bottom-up*: adiciona uma feature por vez | $\arg\max_l J[\hat{\Xi} \cup \{\xi_l\}]$ |
| **SBS** | *Top-down*: remove uma feature por vez | $\arg\max_l J[\hat{\Xi} \setminus \{\xi_l\}]$ |
| **SFFS** | Floating Forward: inclui + tenta excluir | Combina critérios de inclusão e exclusão |
| **SBFS** | Floating Backward: exclui + tenta reincluir | Combina critérios de inclusão e exclusão |

In [ ]:
# SFS e SBS usando o RF como critério de separabilidade
rf_sel = RandomForestClassifier(n_estimators=30, random_state=42)

sfs = SequentialFeatureSelector(rf_sel, n_features_to_select=5,
                                  direction='forward', cv=3, n_jobs=-1)
sbs = SequentialFeatureSelector(rf_sel, n_features_to_select=5,
                                  direction='backward', cv=3, n_jobs=-1)

sfs.fit(X_tr, y_tr)
sbs.fit(X_tr, y_tr)

feat_sfs = np.where(sfs.get_support())[0]
feat_sbs = np.where(sbs.get_support())[0]

print(f'SFS — atributos selecionados (5/{X_tr.shape[1]}): {feat_sfs + 1}')
print(f'SBS — atributos selecionados (5/{X_tr.shape[1]}): {feat_sbs + 1}')

rf_sfs = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sbs = RandomForestClassifier(n_estimators=100, random_state=42)

rf_sfs.fit(X_tr[:, feat_sfs], y_tr)
rf_sbs.fit(X_tr[:, feat_sbs], y_tr)

print(f'\nRF (todos atributos) : {accuracy_score(y_te, rf_orig.predict(X_te)):.4f}')
print(f'RF + SFS (5 atrib.)  : {accuracy_score(y_te, rf_sfs.predict(X_te[:, feat_sfs])):.4f}')
print(f'RF + SBS (5 atrib.)  : {accuracy_score(y_te, rf_sbs.predict(X_te[:, feat_sbs])):.4f}')

---
## 6. Avaliação do Random Forest

### 6.1 Métricas Multiclasse

In [ ]:
rf_eval = RandomForestClassifier(n_estimators=100, random_state=42)
rf_eval.fit(X_train, y_train)
y_pred = rf_eval.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
oa = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

# Acurácia do produtor e do usuário por classe
prod_acc = cm.diagonal() / cm.sum(axis=1)   # sensibilidade por classe
user_acc = cm.diagonal() / cm.sum(axis=0)   # precisão por classe

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, colorbar=False)
ax.set_title(f'Matriz de Confusão\nOA = {oa:.4f} | Kappa = {kappa:.4f}')
plt.tight_layout()
plt.show()

print(f'\nOverall Accuracy (OA): {oa:.4f}')
print(f'Coeficiente Kappa    : {kappa:.4f}')
print(f"\n{'Classe':<10} {'Prod. Acc (Recall)':>20} {'User Acc (Precision)':>22}")
print('-' * 55)
for cls in range(3):
    print(f"{'ω'+str(cls+1):<10} {prod_acc[cls]:>20.4f} {user_acc[cls]:>22.4f}")

### 6.2 Métricas Binárias — Fβ e MCC

$$F_\beta = (1 + \beta^2) \cdot \frac{Pr \cdot Re}{\beta^2 \cdot Pr + Re}$$

$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

In [ ]:
# Problema binário
X_bin, y_bin = make_classification(n_samples=400, n_features=10,
                                    n_classes=2, random_state=42)
X_btr, X_bte, y_btr, y_bte = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42)

rf_bin = RandomForestClassifier(n_estimators=100, random_state=42)
rf_bin.fit(X_btr, y_btr)
y_bpred = rf_bin.predict(X_bte)

mcc = matthews_corrcoef(y_bte, y_bpred)

print('Avaliação Binária — Random Forest')
print('-' * 40)
print(f"F1-Score (β=1)   : {f1_score(y_bte, y_bpred):.4f}")
print(f"F2-Score (β=2)   : {f1_score(y_bte, y_bpred, beta=2):.4f}"  .replace('beta=2', '').replace('f1_score', ''))
from sklearn.metrics import fbeta_score
print(f"F2-Score (β=2)   : {fbeta_score(y_bte, y_bpred, beta=2):.4f}")
print(f"F0.5-Score (β=½) : {fbeta_score(y_bte, y_bpred, beta=0.5):.4f}")
print(f"MCC              : {mcc:.4f}")
print(f"\nInterpretação MCC: {'bom' if mcc > 0.7 else 'moderado' if mcc > 0.4 else 'fraco'}")

---
## 7. Parametrização do Random Forest

### 7.1 Hiperparâmetros principais

| Parâmetro | Descrição | Valor típico |
|---|---|---|
| `n_estimators` (L) | Número de árvores | 100–500 |
| `max_features` | Atributos por nó | `sqrt` (classif.) / `1/3` (regress.) |
| `min_samples_leaf` (ψ) | Mínimo de amostras por folha | 1–20 |
| `min_impurity_decrease` (ζ) | Redução mínima de impureza | 0.0 |
| `max_depth` | Profundidade máxima | None (ilimitado) |

### 7.2 Grid Search + Cross-Validation v-fold

In [ ]:
param_grid = {
    'n_estimators'     : [50, 100, 200],
    'max_features'     : ['sqrt', 'log2'],
    'min_samples_leaf' : [1, 5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs.fit(X_train, y_train)

print('--- Grid Search ---')
print(f'Melhores parâmetros : {gs.best_params_}')
print(f'Acurácia CV (média) : {gs.best_score_:.4f}')
print(f'Acurácia no teste   : {accuracy_score(y_test, gs.best_estimator_.predict(X_test)):.4f}')

### 7.3 Randomized Search

In [ ]:
from scipy.stats import randint

param_dist = {
    'n_estimators'     : randint(50, 300),
    'max_features'     : ['sqrt', 'log2'],
    'min_samples_leaf' : randint(1, 20),
    'max_depth'        : [None, 5, 10, 20],
}

rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=20,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
rs.fit(X_train, y_train)

print('--- Randomized Search ---')
print(f'Melhores parâmetros : {rs.best_params_}')
print(f'Acurácia CV (média) : {rs.best_score_:.4f}')
print(f'Acurácia no teste   : {accuracy_score(y_test, rs.best_estimator_.predict(X_test)):.4f}')

In [ ]:
# Comparação Grid Search vs Randomized Search
import pandas as pd

gs_results = pd.DataFrame(gs.cv_results_)[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5)
rs_results = pd.DataFrame(rs.cv_results_)[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5)

print('Top-5 Grid Search:')
print(gs_results.to_string(index=False))
print(f'\nTop-5 Randomized Search:')
print(rs_results.to_string(index=False))

---
## 8. Random Forest para Regressão

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

X_r, y_r = make_regression(n_samples=300, n_features=10, n_informative=5, noise=20, random_state=42)
X_rtr, X_rte, y_rtr, y_rte = train_test_split(X_r, y_r, test_size=0.3, random_state=42)

modelos_reg = {
    'CART Regressão'       : DecisionTreeRegressor(random_state=42),
    'Bagging Regressão'    : BaggingClassifier(estimator=DecisionTreeRegressor(),
                                               n_estimators=50, random_state=42),
    'Random Forest Regr.'  : RandomForestRegressor(n_estimators=100, random_state=42),
}

print(f"{'Modelo':<25} {'RMSE':>10} {'R²':>10}")
print('-' * 47)
for nome, clf in modelos_reg.items():
    try:
        clf.fit(X_rtr, y_rtr)
        pred = clf.predict(X_rte)
        rmse = np.sqrt(mean_squared_error(y_rte, pred))
        r2   = r2_score(y_rte, pred)
        print(f"{nome:<25} {rmse:>10.4f} {r2:>10.4f}")
    except Exception:
        print(f"{nome:<25} N/A (requer regressor)")

# Correto para Bagging Regressão
from sklearn.ensemble import BaggingRegressor
bag_r = BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=50, random_state=42)
bag_r.fit(X_rtr, y_rtr)
pred_b = bag_r.predict(X_rte)
print(f"{'Bagging Regressão':<25} {np.sqrt(mean_squared_error(y_rte, pred_b)):>10.4f} {r2_score(y_rte, pred_b):>10.4f}")

---
## 9. Pipeline Completo: Pré-processamento → RF → Avaliação

In [ ]:
from sklearn.pipeline import Pipeline

# Dataset de alta dimensão
X_full, y_full = make_classification(n_samples=500, n_features=50, n_informative=10,
                                      n_redundant=10, n_classes=3, random_state=42)
X_ftr, X_fte, y_ftr, y_fte = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

# Pipeline 1: RF direto
pipe_rf = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Pipeline 2: PCA + RF
pipe_pca_rf = Pipeline([
    ('pca', PCA(n_components=0.95)),
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42))
])

# Pipeline 3: Escala + PCA + RF
pipe_full = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=0.95)),
    ('rf',     RandomForestClassifier(n_estimators=100, random_state=42))
])

print(f"{'Pipeline':<30} {'Acurácia':>10} {'Kappa':>10}")
print('-' * 52)
for nome, pipe in [('RF direto', pipe_rf), ('PCA + RF', pipe_pca_rf), ('Escala + PCA + RF', pipe_full)]:
    pipe.fit(X_ftr, y_ftr)
    pred = pipe.predict(X_fte)
    acc  = accuracy_score(y_fte, pred)
    kap  = cohen_kappa_score(y_fte, pred)
    print(f"{nome:<30} {acc:>10.4f} {kap:>10.4f}")

---
## 10. Resumo — Mapa Completo do Random Forest

```
┌─────────────────────────────────────────────────────────────────────┐
│                        RANDOM FOREST                                │
├─────────────────────────────────────────────────────────────────────┤
│  PRÉ-PROCESSAMENTO                                                  │
│  ├─ PCA / LLE       (redução dimensional)                           │
│  ├─ SFS / SBS       (seleção sequencial de atributos)               │
│  └─ SFFS / SBFS     (seleção flutuante)                             │
├─────────────────────────────────────────────────────────────────────┤
│  GERAÇÃO DE DIVERSIDADE                                             │
│  ├─ Bootstrap       (amostragem com reposição por árvore)           │
│  └─ Atributos aleatórios (√p ou log₂p atributos por nó)            │
├─────────────────────────────────────────────────────────────────────┤
│  BASE LEARNER — CART                                                │
│  ├─ Classificação   (Entropia de Informação, folha = classe maioria)│
│  ├─ Regressão       (Desvio Quadrático, folha = média)              │
│  └─ Parâmetros: ζ (min impureza), ψ (min amostras/folha)           │
├─────────────────────────────────────────────────────────────────────┤
│  COMBINAÇÃO                                                         │
│  ├─ Majority Voting (classificação — hard)                          │
│  ├─ Regra da Soma   (classificação — soft, com pesos)               │
│  └─ Média           (regressão)                                     │
├─────────────────────────────────────────────────────────────────────┤
│  ANÁLISE INTERNA                                                    │
│  ├─ OOB Estimation  (erro de generalização sem val. separado)       │
│  └─ Importância de atributos (redução média de impureza)            │
├─────────────────────────────────────────────────────────────────────┤
│  META-ENSEMBLE                                                      │
│  └─ Stacking        (RF como base learner + meta-classificador)     │
├─────────────────────────────────────────────────────────────────────┤
│  PARAMETRIZAÇÃO                                                     │
│  ├─ Grid Search + v-fold Cross-Validation (exaustivo)               │
│  └─ Randomized Search (eficiente, exploração estocástica)           │
├─────────────────────────────────────────────────────────────────────┤
│  AVALIAÇÃO                                                          │
│  ├─ Multiclasse: OA, Kappa, Tau, Prod./User. Accuracy              │
│  └─ Binária:     F-beta (β=1,2,½), MCC                             │
└─────────────────────────────────────────────────────────────────────┘
```

---
*Análise elaborada com base nos materiais da disciplina de Reconhecimento de Padrões — UNESP ICT, Prof. Dr. Rogério Galante Negri.*